In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
import numpy as np, pandas as pd
ALPHA=config.ALPHA_PRIMARY; R=getattr(config,'N_MATCHED_DRAWS',10)
RNG=np.random.default_rng(20260726)
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
print('ready:', os.getcwd(), '| alpha', ALPHA)


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research | alpha 0.05


In [2]:
# =============================================================================
# Cell 2 - SELECTIVE PREDICTION POLICY (fully label-free).
#
# The monitor tells an operator WHICH CLASSES will undercover, but not what to
# do about it. The natural operational answer is to abstain: route the least
# trustworthy alerts to a human analyst and keep the conformal guarantee on the
# remainder. The question the paper can then answer is the one an operator
# actually asks: how much analyst effort buys back nominal coverage?
#
# Per-alert abstention score, using ONLY scores and source-calibrated quantiles:
#     margin(x) = max_c ( q_c - s(x, c) )
# margin >= 0  -> some class sits inside its quantile (non-empty prediction set)
# margin <  0  -> empty set, i.e. a guaranteed miss
# Abstaining in ascending margin escalates the most atypical alerts first. The
# true label is never consulted, so the policy is deployable.
#
# We report, at each escalation rate: focal coverage among RETAINED alerts, and
# the fraction of focal-class alerts retained. The second is a guard: coverage
# on a remnant is meaningless if abstention simply deleted the class.
# =============================================================================
def aps_scores(P, rng):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

FRACS=np.round(np.arange(0.0,0.51,0.025),3)   # escalate 0% .. 50% of alerts

def selective_curve(S_eval, y_eval, q, focal_idx):
    """S_eval: per-class scores; q: per-class source quantiles; label-free abstention."""
    margin=(q[None,:]-S_eval).max(axis=1)
    order=np.argsort(margin, kind='stable')          # most atypical first
    n=len(margin); focal_mask=(y_eval==focal_idx); n_focal=int(focal_mask.sum())
    covered=(S_eval[np.arange(n),y_eval]<=q[y_eval])  # per-alert coverage indicator
    rows=[]
    for f in FRACS:
        k=int(round(f*n))
        abstain=np.zeros(n,bool); abstain[order[:k]]=True
        keep=~abstain
        fk=keep&focal_mask
        rows.append({'escalated_frac':float(f),
                     'focal_cov_retained':float(covered[fk].mean()) if fk.any() else np.nan,
                     'focal_retained_frac':float(fk.sum()/n_focal) if n_focal else np.nan,
                     'marginal_cov_retained':float(covered[keep].mean()) if keep.any() else np.nan,
                     'empty_set_frac':float((margin<0).mean())})
    return rows
print('selective policy ready | escalation grid:', FRACS[0], '->', FRACS[-1])


selective policy ready | escalation grid: 0.0 -> 0.5


In [3]:
# =============================================================================
# Cell 3 - NSL-KDD at rung 0.80 (the hardest case: focal coverage 0.030).
# Reconstruction matches nb22/nb05. Randomized APS = coverage of record.
# =============================================================================
CLASSES=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CLASSES)}; K=len(CLASSES)
FOCAL='R2L'; FIDX=c2i[FOCAL]; RUNG=0.80
nsl_train=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
nsl_test =pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
nsl_train=nsl_train.assign(partition=part['partition'].values)
y_sp=nsl_train[nsl_train.partition=='source_cal_pool']['label'].map(c2i).to_numpy()
y_te=nsl_test['label'].map(c2i).to_numpy()
assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
REALS=sorted(assign[np.isclose(assign.rung,RUNG)]['realization'].unique())
print(f'NSL rung {RUNG}: {len(REALS)} realizations')

rows=[]
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); P_sp, P_te = d['S_pool'].astype(np.float64), d['target'].astype(np.float64)
    for j in REALS:
        ev=IDX.get((RUNG,j,'eval'))
        if ev is None or len(ev)==0: continue
        rng=np.random.default_rng(dseed('sel-nsl',arch,seed,j))
        S_cal=aps_scores(P_sp, np.random.default_rng(dseed('sel-nsl',arch,seed,j,'c')))
        S_ev =aps_scores(P_te[ev], np.random.default_rng(dseed('sel-nsl',arch,seed,j,'e')))
        tc=S_cal[np.arange(len(y_sp)),y_sp]
        q=np.array([conformal_q(tc[y_sp==c],ALPHA)[0] for c in range(K)])
        if not np.isfinite(q[FIDX]): continue
        for r in selective_curve(S_ev, y_te[ev], q, FIDX):
            r.update({'dataset':'nslkdd','arch':arch,'seed':seed,'realization':j}); rows.append(r)
nsl_sel=pd.DataFrame(rows)
g=nsl_sel.groupby('escalated_frac')[['focal_cov_retained','focal_retained_frac','marginal_cov_retained']].mean()
print('\nNSL-KDD selective prediction (focal R2L, SHC, alpha=0.05):')
print(g.round(4).to_string())


NSL rung 0.8: 20 realizations

NSL-KDD selective prediction (focal R2L, SHC, alpha=0.05):
                focal_cov_retained  focal_retained_frac  marginal_cov_retained
escalated_frac                                                                
0.000                       0.0289               1.0000                 0.6076
0.025                       0.0290               0.9820                 0.6076
0.050                       0.0288               0.9613                 0.6070
0.075                       0.0292               0.9336                 0.6075
0.100                       0.0297               0.9082                 0.6076
0.125                       0.0295               0.8895                 0.6079
0.150                       0.0301               0.8636                 0.6079
0.175                       0.0302               0.8356                 0.6085
0.200                       0.0305               0.8103                 0.6089
0.225                       0.0303       

In [ ]:
# =============================================================================
# Cell 4 - CIC-IDS2017 (focal DoS) and UGR'16 (focal nerisbotnet).
# Same matched-draw structure as nb23/nb29 so the setting is identical.
# =============================================================================
rows=[]
# ---- CIC ----
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=lab_cic(spx),lab_cic(tgx); mm=min(len(spx),len(tgx)//2)
    for f in sorted((config.DATA_DIR/'cic_probs').glob(f'{name}__*.npz')):
        _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
        d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
        for draw in range(R):
            rng=np.random.default_rng(dseed(name,seed,arch,draw))
            tp=rng.permutation(len(ytg)); de=tp[:mm]; sc=rng.permutation(len(ysp))[:mm]
            S_cal=aps_scores(Psp[sc], np.random.default_rng(dseed(name,seed,arch,draw,'sc')))
            S_ev =aps_scores(Ptg[de], np.random.default_rng(dseed(name,seed,arch,draw,'e')))
            ysc=ysp[sc]; tc=S_cal[np.arange(len(ysc)),ysc]
            q=np.array([conformal_q(tc[ysc==c],ALPHA)[0] for c in range(2)])
            if not np.isfinite(q[1]): continue
            for r in selective_curve(S_ev, ytg[de], q, 1):
                r.update({'dataset':'cicids2017','arch':arch,'seed':seed,'realization':name}); rows.append(r)
# ---- UGR ----
UGR=config.DATASETS_DIR/'ugr16'
usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}; FU=U2I['nerisbotnet']
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,qq in zip(nm,c): a.loc[idx[kk:kk+qq]]=a2; kk+=qq
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp_u=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy(); y_tg_u=utgt['label'].map(U2I).to_numpy()
mmU=min(len(y_sp_u),len(y_tg_u)//2)
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        tp=rng.permutation(len(y_tg_u)); de=tp[:mmU]; sc=rng.permutation(len(y_sp_u))[:mmU]
        S_cal=aps_scores(Psp[sc], np.random.default_rng(dseed('ugr16',seed,arch,draw,'s')))
        S_ev =aps_scores(Ptg[de], np.random.default_rng(dseed('ugr16',seed,arch,draw,'e')))
        ysc=y_sp_u[sc]; tc=S_cal[np.arange(len(ysc)),ysc]
        q=np.array([conformal_q(tc[ysc==c],ALPHA)[0] for c in range(len(UCL))])
        if not np.isfinite(q[FU]): continue
        for r in selective_curve(S_ev, y_tg_u[de], q, FU):
            r.update({'dataset':'ugr16','arch':arch,'seed':seed,'realization':'july_to_august'}); rows.append(r)
cu_sel=pd.DataFrame(rows)
for ds in ['cicids2017','ugr16']:
    g=cu_sel[cu_sel.dataset==ds].groupby('escalated_frac')[['focal_cov_retained','focal_retained_frac','marginal_cov_retained']].mean()
    print(f'\n{ds} selective prediction:'); print(g.round(4).to_string())


In [ ]:
# =============================================================================
# Cell 5 - HEADLINE: escalation rate required to restore nominal focal coverage,
# with the retention guard. Plus the risk-coverage figure.
# =============================================================================
sel=pd.concat([nsl_sel,cu_sel],ignore_index=True)
summary=sel.groupby(['dataset','escalated_frac'])[
    ['focal_cov_retained','focal_retained_frac','marginal_cov_retained','empty_set_frac']].mean().reset_index()

NOMINAL=1-ALPHA; GUARD=0.20     # require >=20% of focal alerts still retained
head=[]
for ds,g in summary.groupby('dataset'):
    g=g.sort_values('escalated_frac')
    ok=g[(g.focal_cov_retained>=NOMINAL)&(g.focal_retained_frac>=GUARD)]
    best=g.loc[g.focal_cov_retained.idxmax()]
    head.append({'dataset':ds,
        'baseline_focal_cov':round(float(g.iloc[0].focal_cov_retained),4),
        'escalation_to_nominal':(round(float(ok.iloc[0].escalated_frac),3) if len(ok) else None),
        'focal_retained_at_that_point':(round(float(ok.iloc[0].focal_retained_frac),3) if len(ok) else None),
        'best_focal_cov_within_50pct':round(float(best.focal_cov_retained),4),
        'at_escalation':round(float(best.escalated_frac),3),
        'focal_retained_at_best':round(float(best.focal_retained_frac),3),
        'empty_set_frac':round(float(g.iloc[0].empty_set_frac),4)})
H=pd.DataFrame(head)
print(f'SELECTIVE PREDICTION HEADLINE (nominal {NOMINAL:.2f}, retention guard {100*GUARD:.0f}%):')
print(H.to_string(index=False))
print('\nread: escalation_to_nominal is the fraction of ALL alerts a human must review')
print('      for the conformal guarantee to hold on what remains. None = not reachable')
print('      within a 50% escalation budget while retaining enough focal alerts.')

import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'figure.dpi':300})
C={'nslkdd':'#2f4b7c','cicids2017':'#c65911','ugr16':'#4a5a2f'}
L={'nslkdd':'NSL-KDD (R2L)','cicids2017':'CIC-IDS2017 (DoS)','ugr16':"UGR'16 (nerisbotnet)"}
fig,axs=plt.subplots(1,2,figsize=(9.6,4.2))
for ds,g in summary.groupby('dataset'):
    g=g.sort_values('escalated_frac')
    axs[0].plot(100*g.escalated_frac,g.focal_cov_retained,marker='o',ms=3.5,color=C[ds],label=L[ds])
    axs[1].plot(100*g.escalated_frac,100*g.focal_retained_frac,marker='o',ms=3.5,color=C[ds],label=L[ds])
axs[0].axhline(NOMINAL,color='#666',ls='--',lw=1,label='nominal 0.95')
axs[0].set_xlabel('alerts escalated to an analyst (%)'); axs[0].set_ylabel('focal coverage on retained alerts')
axs[0].set_title('(a) coverage recovered by abstention'); axs[0].legend(fontsize=8,frameon=False)
axs[1].axhline(100*GUARD,color='#b3261e',ls=':',lw=1,label='retention guard 20%')
axs[1].set_xlabel('alerts escalated to an analyst (%)'); axs[1].set_ylabel('focal alerts retained (%)')
axs[1].set_title('(b) how much of the focal class survives'); axs[1].legend(fontsize=8,frameon=False)
fig.tight_layout(); fig.savefig(config.REPORTS_DIR/'selective_prediction.png',bbox_inches='tight',facecolor='white')
print('\nfigure saved: selective_prediction.png')


In [ ]:
# =============================================================================
# Cell 6 - save + commit
# =============================================================================
summary.to_csv(config.REPORTS_DIR/'selective_prediction_curve.csv',index=False)
H.to_csv(config.REPORTS_DIR/'selective_prediction_headline.csv',index=False)
(config.REPORTS_DIR/'selective_prediction_verdict.json').write_text(json.dumps({
 'analysis':'label-free selective prediction: abstain on low-margin alerts, restore coverage on the remainder',
 'policy':'margin(x)=max_c(q_c - s(x,c)); escalate ascending margin; true label never used',
 'nominal':NOMINAL,'retention_guard':GUARD,'headline':H.to_dict('records'),
 'reads':('escalation_to_nominal answers the operator question the monitor alone cannot: how much '
          'analyst effort buys back the conformal guarantee. focal_retained_frac guards against the '
          'degenerate solution of abstaining the focal class out of existence.')},indent=2,default=str))
print('saved 2 CSVs + verdict JSON + figure')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb30: label-free selective prediction - risk-coverage tradeoff, escalation rate to restore nominal focal coverage')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
